# Wan 2.2 NF4 — Colab T4
Install dependencies before importing Diffusers. Restart session if dependency detection is stale. 480 square, 17 frames; higher exports are upscaled. Select a job explicitly; smoke is optional after standalone verification.


In [ ]:
%pip install -q diffusers==0.42.1 transformers==5.22.1 bitsandbytes==0.50.2 accelerate sentencepiece ftfy "imageio[ffmpeg]"

In [ ]:
# 1. Verify the GPU (Runtime > Change runtime type > T4 GPU)
import torch
assert torch.cuda.is_available(), 'No GPU — enable T4: Runtime > Change runtime type'
print(torch.cuda.get_device_name(0))
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 2. Mount Google Drive and point at the exchange folder
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
EXCHANGE = Path('/content/drive/MyDrive/heritage-exchange')
assert (EXCHANGE / 'pipeline_gpu').is_dir(), (
    'Exchange folder not found. Run `python -m pipeline.run --stage package` '
    'locally, then sync/upload the exchange dir to My Drive as heritage-exchange.'
)

In [ ]:
# 4. Import the synced generation module
import sys
sys.path.insert(0, str(EXCHANGE / 'pipeline_gpu'))
import gpu_video

In [ ]:
# Optional smoke test (skip if already verified)
RUN_SMOKE = False
if RUN_SMOKE:
    print(gpu_video.run_smoke(EXCHANGE))

In [ ]:
jobs = sorted((EXCHANGE / 'jobs').glob('video-*/job.json'))
print([p.parent.name for p in jobs])
JOB_ID = ''  # paste intended job ID
if JOB_ID:
    entry = gpu_video.run_job(EXCHANGE / 'jobs' / JOB_ID / 'job.json', EXCHANGE)
    print(entry)

## Done

Results are in `heritage-exchange/results/`. Wait for Drive to sync them to
your machine (or download the folder manually), then run locally:

```bash
python -m pipeline.run --resume
```

Shots that failed on GPU are covered by the ffmpeg keyframe fallback during
assemble — check `runs.jsonl` for `status: fallback` entries.